# End-to-end evaluation on the rebuilt stack

Every generation metric this project has published is now stale. Faithfulness 4.90, relevancy 4.80, correct-article rate 35/40 — all measured on Cohere embeddings, clause-level chunks, `top_k=10`, and the 40-question v1 eval set. **Not one of those inputs still exists.**

This is a single fresh measurement of the system as it will actually ship.

| | was | now |
|---|---|---|
| embeddings | Cohere `embed-v4.0` | **Gemini `gemini-embedding-001`** |
| chunks | 739 clause fragments | **531 whole articles** |
| `top_k` | 10 | **5** |
| RRF weight | 0.85 | **0.90** |
| eval set | v1, 40 questions, statute-worded | **v3, 60 questions, natural phrasing** |
| system prompt | Phase 7, five rules | **Phase 20, rewritten for this stack** |

## One arm, deliberately

This does **not** compare the new prompt against the old one. Measuring a prompt written for a system that no longer exists would characterise a configuration that will never ship, and the old prompt's known failure modes were diagnosed on clause chunks and statute-shaped questions.

The consequence is stated plainly rather than buried: **this run cannot claim the new prompt is better than the old.** It establishes where the system stands now. Any future prompt change gets compared against *these* numbers.

## What is measured

Retrieval and generation are reported as **separate sections**, never blended into one table — they answer different questions and have different failure modes.

- **Retrieval** — recall@1/@5, MRR, rank distribution. Deterministic.
- **Generation, regex proxy** — did the answer cite the correct article, and did it cite any article that was *not* in the context it was given (fabrication)? Cheap and objective, but a proxy: citing the right number is not the same as being right.
- **Generation, LLM judge** — faithfulness and relevancy, 1–5, by Claude Sonnet via OpenRouter. This is the metric that matters; the regex check exists to catch what the judge might wave through.

## Persistence

Retrieved chunks, raw answers and judge scores are each written to disk as they are produced, and the notebook resumes from what is already there.

This is not incidental. Phase 8 saved only scores, so when `ARTICLE_NUM_RE` turned out to be broken — it anchored on `المادة`, which does not occur inside `للمادة` because Arabic elides the alef when ل joins ال — six correct Arabic citations had been scored as misses and the run could not be re-scored without re-generating. Phase 12b then lost three runs to OpenRouter 402s before persistence was added. Both lessons are applied here from the start.

In [1]:
import json
import os
import re as _re
import sys
import time
from pathlib import Path

import pandas as pd

PROJECT = Path(r"c:\Users\Mohannad Alsaleh\Desktop\Data Science\Data Science Projects\1st RAG project")
sys.path.insert(0, str(PROJECT))

from src.retrieval import Retriever, EMBED_PROVIDER, PINECONE_INDEX_NAME, RRF_W, RRF_K
from src.generation import (get_production_generator, SYSTEM_PROMPT, CONTEXT_SYSTEM_PROMPT,
                            ARTICLE_NUM_RE, LAW_DISPLAY_NAMES)

EVAL_PATH = PROJECT / "eval" / "eval_v3.jsonl"
RETR_PATH = PROJECT / "eval" / "final_eval_v3_retrieval.csv"
GEN_PATH = PROJECT / "eval" / "final_eval_v3_generations.csv"
JUDGE_PATH = PROJECT / "eval" / "final_eval_v3_judge.csv"

ev = pd.DataFrame([json.loads(l) for l in open(EVAL_PATH, encoding="utf-8")])

print(f"eval: {len(ev)} questions ({EVAL_PATH.name})")
print(f"retrieval: {EMBED_PROVIDER} -> {PINECONE_INDEX_NAME}, RRF k={RRF_K} w={RRF_W}")
print(f"system prompt: {len(SYSTEM_PROMPT)} chars")
print("\nfirst 3 lines of the prompt under test:")
for line in SYSTEM_PROMPT.splitlines()[:3]:
    print("   ", line[:100])

eval: 60 questions (eval_v3.jsonl)
retrieval: gemini -> jordan-labor-law-articles-gemini, RRF k=5 w=0.9
system prompt: 1584 chars

first 3 lines of the prompt under test:
    You are a legal information assistant for Jordanian labour and social security law (the Labour Law 1
    
    Grounding


## 1. Retrieval

Uses the production `Retriever` unchanged — same index, same fusion, same `top_k` default. The retrieved chunks are saved, so generation and judging can be re-run later without paying for retrieval again.

In [2]:
if RETR_PATH.exists():
    retr = pd.read_csv(RETR_PATH, encoding="utf-8-sig")
    retr["chunk_ids"] = retr["chunk_ids"].apply(json.loads)
    print(f"loaded cached retrieval for {len(retr)} questions")
else:
    r = Retriever()
    rows = []
    for q in ev.itertuples(index=False):
        chunks = r.retrieve(q.question_text)          # top_k comes from the production default
        want = {str(a) for a in q.answer_article_numbers}
        rank, found = None, set()
        for pos, c in enumerate(chunks, start=1):
            if (c["law"] == q.answer_law and c["language"] == q.answer_language
                    and c["article_number"] in want):
                if rank is None:
                    rank = pos
                found.add(c["article_number"])
        rows.append({
            "question_id": q.question_id,
            "chunk_ids": [c["chunk_id"] for c in chunks],
            "rank": rank,
            "n_found": len(found),
            "full_coverage": found == want,
        })
        print(f"  retrieved {len(rows)}/{len(ev)}", end="\r")
    retr = pd.DataFrame(rows)
    out = retr.copy()
    out["chunk_ids"] = out["chunk_ids"].apply(json.dumps)
    out.to_csv(RETR_PATH, index=False, encoding="utf-8-sig")
    print(f"\nsaved -> {RETR_PATH.name}")

chunks_tbl = pd.read_parquet(PROJECT / "0. cleaned_data" / "chunks_articles.parquet")
chunks_tbl["article_number"] = chunks_tbl["article_number"].astype(str)
BY_ID = {c.chunk_id: c for c in chunks_tbl.itertuples()}


def chunks_for(qid):
    ids = retr.loc[retr.question_id == qid, "chunk_ids"].iloc[0]
    return [{"chunk_id": i, "law": BY_ID[i].law, "language": BY_ID[i].language,
             "article_number": BY_ID[i].article_number, "text": BY_ID[i].text} for i in ids]

loaded cached retrieval for 60 questions


## 2. Generation

Production generator (`gpt-4o-mini` via OpenRouter) with the Phase 20 prompt. Answers are flushed to disk after every question and the loop resumes from what is already saved — so a crash, a rate limit or a scoring bug never costs a second generation pass.

In [3]:
gen_rows = []
if GEN_PATH.exists():
    gen_rows = pd.read_csv(GEN_PATH, encoding="utf-8-sig").to_dict("records")
done = {r["question_id"] for r in gen_rows}
print(f"{len(done)} answers already saved, {len(ev) - len(done)} to generate")

if len(done) < len(ev):
    generator = get_production_generator()
    for q in ev.itertuples(index=False):
        if q.question_id in done:
            continue
        ch = chunks_for(q.question_id)
        res = generator.generate(q.question_text, ch)
        gen_rows.append({
            "question_id": q.question_id,
            "question_text": q.question_text,
            "question_language": q.question_language,
            "answer": res["answer"],                       # raw text persisted -- Phase 8 lesson
            "citations": json.dumps(res["citations"], ensure_ascii=False),
        })
        pd.DataFrame(gen_rows).to_csv(GEN_PATH, index=False, encoding="utf-8-sig")
        print(f"  generated {len(gen_rows)}/{len(ev)}", end="\r")

gen = pd.DataFrame(gen_rows)
print(f"\n{len(gen)} answers ready")

60 answers already saved, 0 to generate

60 answers ready


## 3. Regex proxy — correct citation and fabrication

Two objective checks over the saved answer text:

- **correct article cited** — does the answer name an article the ground truth lists?
- **fabricated citation** — does it name an article that was *not* in the context it was given? This is the failure Phase 8 caught twice (q09, q21), where a correct citation sat next to an invented one.

Because the answers are on disk, both can be re-scored for free if the pattern ever changes again — which it did once already.

In [4]:
def cited_articles(answer):
    return {g1 or g2 for g1, g2 in ARTICLE_NUM_RE.findall(str(answer))}


rows = []
for q in ev.itertuples(index=False):
    ans = gen.loc[gen.question_id == q.question_id, "answer"].iloc[0]
    ch = chunks_for(q.question_id)
    in_context = {c["article_number"] for c in ch}
    want = {str(a) for a in q.answer_article_numbers}
    cited = cited_articles(ans)
    rows.append({
        "question_id": q.question_id,
        "cited": sorted(cited),
        "correct_article_cited": bool(cited & want),
        "all_correct_articles_cited": want <= cited,
        "fabricated": sorted(cited - in_context),
        "has_fabrication": bool(cited - in_context),
        "answer_chars": len(str(ans)),
    })
proxy = pd.DataFrame(rows)
n = len(proxy)
print(f"correct article cited      : {proxy.correct_article_cited.sum()}/{n} "
      f"({proxy.correct_article_cited.mean():.1%})")
print(f"all correct articles cited : {proxy.all_correct_articles_cited.sum()}/{n}")
print(f"answers with a fabricated citation: {proxy.has_fabrication.sum()}/{n}")
if proxy.has_fabrication.any():
    print("\n  ", proxy[proxy.has_fabrication][["question_id", "cited", "fabricated"]].to_string(index=False))
print(f"\nanswer length: median {proxy.answer_chars.median():.0f} chars, "
      f"max {proxy.answer_chars.max()}")

correct article cited      : 55/60 (91.7%)
all correct articles cited : 51/60
answers with a fabricated citation: 1/60

   question_id    cited fabricated
        q34 [26, 28]       [28]

answer length: median 448 chars, max 1074


## 4. LLM judge — faithfulness and relevancy

Claude Sonnet via OpenRouter, scoring 1–5 on each. Scores are flushed per call and the loop resumes, for the reasons in the header.

`max_tokens` must be set, and set to a *middle* value. Both extremes have now failed in this project:

- **Too low (300, Phase 12b)** — Claude's internal reasoning consumed the whole budget and the call returned empty content.
- **Unset (this notebook's first run)** — OpenRouter reserves credit against the request's `max_tokens` **ceiling**, not its actual usage. Unset defaults to the model's maximum of 65,536, so each call demanded credit for 65,536 tokens to produce a ~200-token verdict, and 402'd with ~60,000 tokens of balance still available. The run died at question 31 of 60 and the retries were futile — the ceiling never changed between attempts. Every judged question was a Social Security Law question, so the aborted half was not a random half.

`JUDGE_MAX_TOKENS = 2000` sits well above the verdict length and roughly 33x below the ceiling that caused the 402. The guard below still reports `finish_reason` on empty content rather than dying on a `None`.

In [5]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(PROJECT / ".env")
judge_client = OpenAI(api_key=os.environ["OPENROUTER_API_KEY"],
                      base_url="https://openrouter.ai/api/v1")
JUDGE_MODEL = "anthropic/claude-sonnet-5"
JUDGE_MAX_TOKENS = 2000  # see the note above -- unset means a 65,536-token credit reservation

JUDGE_PROMPT = """You are grading an AI assistant's answer to a legal question about Jordanian labor/social security law. Score it on two dimensions, each 1-5.

FAITHFULNESS (1-5): Is every factual claim in the answer actually supported by the source documents below? 5 = every claim is directly backed by the documents, nothing added. 3 = mostly supported but with some unsupported/embellished detail. 1 = contains claims that contradict or aren't found anywhere in the documents.

RELEVANCY (1-5): Does the answer directly and completely address the question asked? 5 = fully addresses it, on-topic, complete. 3 = addresses it but incomplete or padded with tangential info. 1 = doesn't actually answer the question.

Question: {question}

Source documents given to the assistant:
{context}

Assistant's answer:
{answer}

Respond with ONLY this JSON, no other text:
{{"faithfulness_score": <1-5>, "faithfulness_reason": "<one sentence>", "relevancy_score": <1-5>, "relevancy_reason": "<one sentence>"}}"""


def judge_one(question, chunks, answer, max_retries=5):
    ctx = "\n\n".join(f"[{c['law']} Article {c['article_number']} ({c['language']})]\n{c['text']}"
                      for c in chunks)
    last = None
    for attempt in range(max_retries):
        try:
            r = judge_client.chat.completions.create(
                model=JUDGE_MODEL,
                max_tokens=JUDGE_MAX_TOKENS,
                messages=[{"role": "user", "content": JUDGE_PROMPT.format(
                    question=question, context=ctx, answer=answer)}])
            if r.choices:
                content = r.choices[0].message.content
                if not content:
                    last = f"empty content (finish_reason={r.choices[0].finish_reason})"
                else:
                    m = _re.search(r"\{.*\}", content, _re.DOTALL)
                    if m:
                        return json.loads(m.group(0))
                    last = f"no JSON in: {content[:200]!r}"
        except Exception as e:
            last = f"{type(e).__name__}: {str(e)[:200]}"
        wait = min(30, 2 ** attempt)
        print(f"    judge retry {attempt + 1}/{max_retries}: {last} -- waiting {wait}s")
        time.sleep(wait)
    raise RuntimeError(f"judge failed: {last}")


judge_rows = []
if JUDGE_PATH.exists():
    judge_rows = pd.read_csv(JUDGE_PATH, encoding="utf-8-sig").to_dict("records")
jdone = {r["question_id"] for r in judge_rows}
print(f"{len(jdone)} already judged, {len(ev) - len(jdone)} to go")

for q in ev.itertuples(index=False):
    if q.question_id in jdone:
        continue
    ans = gen.loc[gen.question_id == q.question_id, "answer"].iloc[0]
    verdict = judge_one(q.question_text, chunks_for(q.question_id), ans)
    judge_rows.append({"question_id": q.question_id, **verdict})
    pd.DataFrame(judge_rows).to_csv(JUDGE_PATH, index=False, encoding="utf-8-sig")
    print(f"  judged {len(judge_rows)}/{len(ev)}", end="\r")

jdg = pd.DataFrame(judge_rows)
print(f"\n{len(jdg)} judged")

60 already judged, 0 to go

60 judged


## Results

Retrieval and generation reported separately.

In [6]:
full = (ev.merge(retr, on="question_id")
          .merge(proxy, on="question_id")
          .merge(jdg, on="question_id"))
full.to_csv(PROJECT / "eval" / "final_eval_v3_summary.csv", index=False, encoding="utf-8-sig")

r_ = full["rank"]
N = len(full)
print("=" * 72)
print("RETRIEVAL   (deterministic -- Gemini + BM25, RRF w=0.90, top_k=5)")
print("=" * 72)
print(f"  recall@1          {(r_ == 1).sum() / N:.3f}")
print(f"  recall@5          {(r_.notna()).sum() / N:.3f}")
print(f"  MRR               {(1 / r_.dropna()).sum() / N:.3f}")
print(f"  full coverage     {full.full_coverage.sum() / N:.3f}   (all correct articles present)")
print(f"  worst rank        {r_.max():.0f}   misses {int(r_.isna().sum())}")

print()
print("=" * 72)
print("GENERATION -- regex proxy (objective, but a proxy)")
print("=" * 72)
print(f"  correct article cited        {full.correct_article_cited.sum()}/{N} "
      f"({full.correct_article_cited.mean():.1%})")
print(f"  all correct articles cited   {full.all_correct_articles_cited.sum()}/{N}")
print(f"  answers with a fabrication   {full.has_fabrication.sum()}/{N}")

print()
print("=" * 72)
print("GENERATION -- LLM judge (Claude Sonnet, 1-5)")
print("=" * 72)
for col in ["faithfulness_score", "relevancy_score"]:
    s = full[col]
    print(f"  {col.replace('_score',''):14s} mean {s.mean():.3f}  median {s.median():.1f}  "
          f"min {s.min()}  |  " + "  ".join(f"{v}:{(s == v).sum()}" for v in [5, 4, 3, 2, 1]))

print()
print("breakdown by question language")
print(full.groupby("question_language")[["faithfulness_score", "relevancy_score"]]
      .agg(["mean", "min"]).round(2).to_string())
print()
print("breakdown by law")
print(full.groupby("answer_law")[["faithfulness_score", "relevancy_score"]]
      .mean().round(2).to_string())

RETRIEVAL   (deterministic -- Gemini + BM25, RRF w=0.90, top_k=5)
  recall@1          0.867
  recall@5          1.000
  MRR               0.922
  full coverage     0.950   (all correct articles present)
  worst rank        4   misses 0

GENERATION -- regex proxy (objective, but a proxy)
  correct article cited        55/60 (91.7%)
  all correct articles cited   51/60
  answers with a fabrication   1/60

GENERATION -- LLM judge (Claude Sonnet, 1-5)
  faithfulness   mean 4.700  median 5.0  min 1  |  5:46  4:12  3:1  2:0  1:1
  relevancy      mean 4.550  median 5.0  min 2  |  5:41  4:12  3:6  2:1  1:0

breakdown by question language
                  faithfulness_score     relevancy_score    
                                mean min            mean min
question_language                                           
ar                               4.7   1            4.43   2
en                               4.7   3            4.67   3

breakdown by law
            faithfulness_score  relevan

In [7]:
# Every answer the judge marked below 5 on either axis -- the actual work list.
low = full[(full.faithfulness_score < 5) | (full.relevancy_score < 5)]
print(f"{len(low)} of {N} answers scored below 5 on at least one axis\n")
for r in low.sort_values(["faithfulness_score", "relevancy_score"]).itertuples():
    print(f"--- {r.question_id} [{r.question_language}] {r.answer_law} art "
          f"{','.join(r.answer_article_numbers)}  faith={r.faithfulness_score} rel={r.relevancy_score}"
          f"  (retrieval rank {r.rank})")
    print(f"    Q: {r.question_text[:110]}")
    if r.faithfulness_score < 5:
        print(f"    faith: {r.faithfulness_reason}")
    if r.relevancy_score < 5:
        print(f"    rel  : {r.relevancy_reason}")

26 of 60 answers scored below 5 on at least one axis

--- q16 [ar] ssl art 24  faith=1 rel=2  (retrieval rank 1)
    Q: هل يُخصم من راتبي شيء مقابل التأمين ضد إصابات العمل، أم يدفعه صاحب العمل وحده؟
    faith: The answer claims the amount is deducted from the employee's salary, contradicting Article 24(a) which states the firm alone pays the 2% contribution, and it cites a nonexistent 'Labour Law 1996' source not found in the documents.
    rel  : While it directly addresses the yes/no question about salary deduction, it gives an incorrect answer that misrepresents the source, making it misleading rather than a correct response.
--- q42 [en] labor96 art 70,71  faith=3 rel=5  (retrieval rank 1)
    Q: How long can I stay off work after giving birth, and do I get time during the day for breastfeeding afterwards
    faith: The answer incorrectly states the full ten weeks is after birth, when Article 70 specifies ten weeks total before and after childbirth with only a minimum six weeks aft

## 5. Defect scan over the raw answers

Two checks the judge cannot make, run over the persisted answer text at zero API cost.

**Citation language.** `_format_context` in `src/generation.py` labels every document with `LAW_DISPLAY_NAMES`, which is English-only. An Arabic question therefore gets Arabic article text under an English law heading, and the model cites what it was shown. This is a defect in our code, not in the model — it is measured here rather than assumed.

**Wrong law named.** The regex proxy extracts article *numbers* only, so an answer citing the right number under the wrong law scores as a hit. Checking the law name separately closes that blind spot.

In [8]:
AR_RANGE = ("؀", "ۿ")


def has_arabic(s):
    return any(AR_RANGE[0] <= ch <= AR_RANGE[1] for ch in str(s))


ans_by_id = dict(zip(gen.question_id, gen.answer))
ar_ids = ev.loc[ev.question_language == "ar", "question_id"]

eng_cite = [q for q in ar_ids if _re.search(r"\bArticle\b", str(ans_by_id[q]))]
print(f"Arabic answers citing in English ('Article N'): {len(eng_cite)}/{len(ar_ids)}")
print(f"  {eng_cite}")

wrong_lang = [q for q in ar_ids if not has_arabic(ans_by_id[q])]
print(f"Arabic questions answered wholly in English  : {len(wrong_lang)}  {wrong_lang}")

# Which law does the answer name, vs which law the ground truth says?
LAW_MARKERS = {
    "ssl": ["Social Security", "الضمان الاجتماعي"],
    "labor96": ["Labor Law", "Labour Law", "قانون العمل"],
    "prof19": ["Professional Work", "العمل المهني"],
}
rows = []
for q in ev.itertuples(index=False):
    a = str(ans_by_id[q.question_id])
    named = {law for law, marks in LAW_MARKERS.items() if any(m in a for m in marks)}
    if named and q.answer_law not in named:
        rows.append({"question_id": q.question_id, "truth": q.answer_law,
                     "named_in_answer": sorted(named)})
print(f"\nanswers naming only a law other than the ground-truth law: {len(rows)}")
for r in rows:
    print("  ", r)

Arabic answers citing in English ('Article N'): 8/30
  ['q16', 'q27', 'q44', 'q46', 'q47', 'q48', 'q53', 'q56']
Arabic questions answered wholly in English  : 0  []

answers naming only a law other than the ground-truth law: 3
   {'question_id': 'q03', 'truth': 'ssl', 'named_in_answer': ['labor96']}
   {'question_id': 'q05', 'truth': 'ssl', 'named_in_answer': ['labor96']}
   {'question_id': 'q16', 'truth': 'ssl', 'named_in_answer': ['labor96']}


## Verdict

**This is the reference point. It is not a comparison.** One arm was run deliberately, so nothing here says the Phase 20 prompt is better or worse than the Phase 7 one — every future change gets measured against these numbers.

### Retrieval — settled

| metric | value |
|---|---|
| recall@1 | 0.867 |
| recall@5 | **1.000** |
| MRR | 0.922 |
| worst rank | 4 |
| misses | 0 |
| full coverage | 0.950 |

Identical recall in both languages (0.867 / 1.000). Reproduces the Phase 19 live check exactly. `top_k=5` is confirmed safe on the full set — nothing landed past rank 4, so the Phase 17 decision holds.

**Retrieval is no longer the bottleneck, and this changes what is worth working on.** In 16 of the 19 questions the judge penalised, the gold article was retrieved at **rank 1**. The information was in front of the model and the answer still lost points. Further retrieval tuning cannot fix any of those.

### Generation — regex proxy

| metric | value |
|---|---|
| correct article cited | 55/60 (91.7%) |
| all correct articles cited | 51/60 |
| flagged fabrications | 1/60 → **0/60 after inspection** |

The single flagged fabrication (q34) is a **false positive**. The answer says the worker is owed wages "unless the termination was a dismissal in accordance with Article 28" — and that clause is quoted from the retrieved Article 26(a), whose own text reads *"unless the termination of the work contract is a dismissal in accordance with article (28) of this Law."* Reproducing a cross-reference contained in the source is the most faithful behaviour available; the proxy penalised it for naming an article that wasn't separately retrieved.

**51 of 531 corpus units cross-reference another article**, so this is a systematic limit of the check, not a one-off. Any future fabrication count should exclude articles named inside retrieved text.

### Generation — LLM judge (Claude Sonnet, 1-5)

| | mean | median | min | 5 / 4 / 3 / 2 / 1 |
|---|---|---|---|---|
| faithfulness | **4.70** | 5.0 | 1 | 46 / 12 / 1 / 0 / 1 |
| relevancy | **4.55** | 5.0 | 2 | 41 / 12 / 6 / 1 / 0 |

By law: labor96 4.77 / 4.77 · prof19 4.75 / 4.75 · **ssl 4.63 / 4.33**. By language: faithfulness 4.70 in both; relevancy 4.43 ar vs 4.67 en.

The Social Security Law is the weak arm on both axes. It is also the longest, most cross-referential of the three laws, which is consistent with the dominant failure below.

### The dominant failure is omission, not hallucination

26 of 60 answers scored below 5. **18 of the 19 relevancy penalties are omission complaints**, and the judge's wording barely varies: "omits funeral expenses" (q01), "omits Article 30" (q05), "omits Article 67(e)" (q19), "omits Article 81" (q29), "omits the 6-month contribution requirement" (q10), "omits the 36-contribution condition" (q25). Four of the 14 faithfulness penalties are the same complaint.

Median answer length by relevancy score: 5 → 445 chars, 4 → 482, 3 → 324, 2 → 258. The lowest-scoring answers are the shortest.

Rule 8 ends with "Keep it brief." Phase 12b's rule 7 — "cover every consequence and exception" — was **removed** because it caused a faithfulness regression (q29, 5 → 3, the model invented a connective claim between two articles). So the two known states of this prompt are *brief and incomplete* or *complete and over-reaching*. The prompt has never been tested in a state that asks for completeness **within** a single provision while still forbidding inference **across** provisions. That is the next thing worth testing, and it is a prompt change, not a retrieval change.

### Two defects that are ours, not the model's

**1. Arabic answers cite in English — 8 of 30 (27%)**: q16, q27, q44, q46, q47, q48, q53, q56. `_format_context` in `src/generation.py` labels every document with `LAW_DISPLAY_NAMES`, which is English-only, so an Arabic question gets Arabic article text under an English heading and the model cites what it was shown. No answer was written wholly in the wrong language — the body is Arabic, the citation is not. Fix is a bilingual display-name map keyed on answer language.

**2. Wrong law named — 3 of 60** (q03, q05, q16), all Social Security Law questions answered from the Labour Law. Both laws were in context in each case. The regex proxy scores all three as **correct**, because it extracts article numbers and never reads the law name.

For q03 and q05 the model's pick is defensible — SSL 27 and labor96 87 both impose injury-reporting duties, and both texts specify 48 hours; SSL 30 and labor96 90 both set death/disability compensation. These look like a ground-truth breadth problem in `eval_v3` as much as a model problem: the question ("what am I required to do straight away?") does not name a law, but the ground truth admits only one.

**q16 is a real failure and the worst answer in the run** (faithfulness 1, relevancy 2). Asked whether work-injury insurance is deducted from the worker's salary, the answer opens **"نعم، يُخصم من راتبك"** — yes, it is deducted — then correctly quotes that the establishment alone pays the 2%. It contradicts itself, wrong part first, and attributes it to the Labour Law. The proxy scored it correct. Note that rule 6 instructs the model to answer the situation "first, in a sentence or two" — committing to a yes/no before working through the article is exactly the shape of this failure, and it is worth checking whether the rule caused it.

### What this run does not measure

- **No comparison arm.** These numbers describe the system; they do not attribute it to the prompt.
- **The judge scores faithfulness to the retrieved context, not correctness against ground truth.** For q03 it wrote that the answer "accurately reflects Article 87(a)" while the ground truth is SSL Article 27 — internally consistent, externally wrong. An answer grounded in the wrong retrieved article can still score 5.
- **Single judge, single run.** Phase 12b measured a real noise floor for judge scores; nothing here is repeated, so small differences against future runs are not readable without re-establishing it.